# Model blending and distribution-shift analysis

## Model blending
Compare the two models on the same 2,000 validation examples, evaluate a small set of blend weights, and produce a Kaggle submission with **25% hybrid MLP + 75% Mia's original ensemble submission**.

In [38]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics import roc_auc_score

In [39]:
# locate directory
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent


sys.path.insert(0, str(project_dir))
from data_utils import load_data

# load validation predictions from both models
hybrid_val = pd.read_csv(project_dir / "hybrid_val_predictions.csv")
mia_val = pd.read_csv(
    project_dir / "improved_ensemble" / "results" / "improved_validation.csv"
)

print("Hybrid MLP:", hybrid_val.shape)
print("Mia's ensemble:", mia_val.shape)

print("\nHybrid columns:", hybrid_val.columns.tolist())
print("Mia's columns:", mia_val.columns.tolist())

# match predictions using passage IDs
comparison = hybrid_val.merge(
    mia_val[["id", "label", "selected_probability_ai"]],
    on="id",
    suffixes=("_hybrid", "_mia"),
    validate="one_to_one"
)

# verify both models were evaluated on the same examples
assert len(comparison) == 2000
assert (comparison["label_hybrid"] == comparison["label_mia"]).all()

print("\nMatched validation examples:", len(comparison))
print("Labels match:", True)
print(comparison.head())


Hybrid MLP: (2000, 3)
Mia's ensemble: (2000, 6)

Hybrid columns: ['id', 'label', 'hybrid_prob']
Mia's columns: ['id', 'label', 'style', 'mlp', 'combined', 'selected_probability_ai']

Matched validation examples: 2000
Labels match: True
     id  label_hybrid  hybrid_prob  label_mia  selected_probability_ai
0  5865             0     0.720152          0                 0.072805
1  1774             1     1.000000          1                 0.996202
2  3336             0     0.002063          0                 0.019326
3  9082             1     0.998575          1                 0.985490
4  6970             0     0.000395          0                 0.083965


In [40]:
# compare individual models and a 50/50 probability blend
y_true = comparison["label_hybrid"].to_numpy()
hybrid_probs = comparison["hybrid_prob"].to_numpy()
mia_probs = comparison["selected_probability_ai"].to_numpy()

# average the probabilities from both models
blend_probs = 0.5 * hybrid_probs + 0.5 * mia_probs

# calculate validation AUROC for each approach
hybrid_auc = roc_auc_score(y_true, hybrid_probs)
mia_auc = roc_auc_score(y_true, mia_probs)
blend_auc = roc_auc_score(y_true, blend_probs)

print(f"Hybrid MLP AUROC:  {hybrid_auc:.6f}")
print(f"Mia ensemble AUROC: {mia_auc:.6f}")
print(f"50/50 blend AUROC:  {blend_auc:.6f}")
print(f"Change vs. Mia:    {blend_auc - mia_auc:+.6f}")


Hybrid MLP AUROC:  0.994118
Mia ensemble AUROC: 0.997794
50/50 blend AUROC:  0.997807
Change vs. Mia:    +0.000013


In [41]:
# compare different weights for the two models
results = []

for hybrid_weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    mia_weight = 1.0 - hybrid_weight

    # weighted average of predicted probabilities
    probs = hybrid_weight * hybrid_probs + mia_weight * mia_probs

    auc = roc_auc_score(y_true, probs)

    results.append({
        "Hybrid Weight": hybrid_weight,
        "Mia Weight": mia_weight,
        "Validation AUROC": auc
    })

blend_results = pd.DataFrame(results)
print(blend_results.to_string(index=False))

 Hybrid Weight  Mia Weight  Validation AUROC
          0.00        1.00          0.997794
          0.25        0.75          0.998052
          0.50        0.50          0.997807
          0.75        0.25          0.996924
          1.00        0.00          0.994118


In [42]:
# load Kaggle test predictions from our hybrid MLP
hybrid_test = pd.read_csv(project_dir / "submission_hybrid_mlp.csv")

# load Mia's original predictions
original_mia = pd.read_csv(
    project_dir / "data" / "submission_improved.csv"
)

# blend with Mia's original Kaggle-scored submission
test_original = hybrid_test.merge(
    original_mia,
    on="id",
    suffixes=("_hybrid", "_mia"),
    validate="one_to_one"
)

# verify all test examples are included
assert len(hybrid_test) == len(original_mia) == len(test_original) == 3000
assert test_original["id"].is_unique

# 25% hybrid MLP + 75% original Mia ensemble
test_original["label"] = (
    0.25 * test_original["label_hybrid"]
    + 0.75 * test_original["label_mia"]
)

# save the blended Kaggle submission
blend_original = test_original[["id", "label"]]
assert blend_original["label"].notna().all()
assert blend_original["label"].between(0, 1).all()

output_path = project_dir / "submission_blend_25_75.csv"
blend_original.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", blend_original.shape)
print(blend_original.head())


Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/submission_blend_25_75.csv
Shape: (3000, 2)
      id     label
0  13001  0.997628
1  13002  0.041700
2  13003  0.083450
3  13004  0.032433
4  13005  0.990829


## Distribution shift and error analysis
Compare labeled training and Kaggle test text distributions, then analyze model performance on held-out examples that appear more test-like or less similar to the original training data. These are diagnostics; they do not establish a causal explanation for the public Kaggle score.


In [43]:
# load the original Kaggle competition dataset
zip_path = (
    project_dir / "data"
    / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip"
)

train, _, test = load_data(zip_path)


# compare text lengths
for df in [train, test]:
    df["word_count"] = df["text"].fillna("").str.split().str.len()
    df["char_count"] = df["text"].fillna("").str.len()

print("TRAIN:")
print(train[["word_count", "char_count"]].describe())

print("\nTEST:")
print(test[["word_count", "char_count"]].describe())

print("\nTRAIN BY LABEL (0=human, 1=AI):")
print(train.groupby("label")[["word_count", "char_count"]].median())


TRAIN:
         word_count    char_count
count  10000.000000  10000.000000
mean     208.708700   1260.909000
std       62.685657    387.461043
min       75.000000    331.000000
25%      160.000000    980.000000
50%      206.000000   1246.000000
75%      256.000000   1522.000000
max      350.000000   2938.000000

TEST:
        word_count   char_count
count  3000.000000  3000.000000
mean    216.972333  1274.930333
std      61.042108   369.962520
min      75.000000   365.000000
25%     173.000000  1025.750000
50%     213.000000  1247.000000
75%     262.000000  1521.000000
max     350.000000  2631.000000

TRAIN BY LABEL (0=human, 1=AI):
       word_count  char_count
label                        
0           204.0      1222.0
1           209.0      1278.0


In [44]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

# combine training and test texts
all_texts = pd.concat([train["text"], test["text"]]).fillna("").to_numpy()

# 0 = training dataset, 1 = Kaggle test dataset
source_labels = np.concatenate([
    np.zeros(len(train)),
    np.ones(len(test))
])

# split before fitting to avoid leakage
X_tr, X_val, y_tr, y_val = train_test_split(
    all_texts,
    source_labels,
    test_size=0.25,
    random_state=42,
    stratify=source_labels
)

# represent texts using word-level TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=30000,
    sublinear_tf=True
)

X_tr_tfidf = vectorizer.fit_transform(X_tr)
X_val_tfidf = vectorizer.transform(X_val)

# predict which dataset each passage came from
domain_model = LogisticRegression(max_iter=1000)
domain_model.fit(X_tr_tfidf, y_tr)

domain_probs = domain_model.predict_proba(X_val_tfidf)[:, 1]
domain_auc = roc_auc_score(y_val, domain_probs)

print(f"Train vs. test distinguishability AUROC: {domain_auc:.4f}")


Train vs. test distinguishability AUROC: 0.7448


In [45]:
# identify words and phrases that distinguish train from test
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = domain_model.coef_[0]

# positive coefficients indicate test-associated terms
top_test = np.argsort(coefficients)[-15:][::-1]

# negative coefficients indicate train-associated terms
top_train = np.argsort(coefficients)[:15]

print("TOP TERMS ASSOCIATED WITH KAGGLE TEST:")
for idx in top_test:
    print(f"{feature_names[idx]:25s} {coefficients[idx]:+.4f}")

print("\nTOP TERMS ASSOCIATED WITH TRAINING:")
for idx in top_train:
    print(f"{feature_names[idx]:25s} {coefficients[idx]:+.4f}")


TOP TERMS ASSOCIATED WITH KAGGLE TEST:
his                       +2.6904
love                      +2.3230
the novel                 +2.2710
novel                     +2.0353
her                       +1.7566
poem                      +1.7237
heart                     +1.6983
the                       +1.6882
in the                    +1.5660
and the                   +1.4990
young                     +1.4580
begins                    +1.4497
it is                     +1.4471
life                      +1.4412
sky                       +1.4257

TOP TERMS ASSOCIATED WITH TRAINING:
was                       -1.8990
film                      -1.8039
overall                   -1.5253
10                        -1.4434
br                        -1.3818
if                        -1.3649
have                      -1.3075
don                       -1.2232
the film                  -1.1451
year                      -1.1092
movie                     -1.1047
using                     -1.0490
fuckin

In [46]:
# train a domain classifier without using our 2000 validation examples
validation_ids = set(comparison["id"])

domain_train = train[~train["id"].isin(validation_ids)]

domain_texts = pd.concat([
    domain_train["text"],
    test["text"]
]).fillna("")

domain_labels = np.concatenate([
    np.zeros(len(domain_train)),
    np.ones(len(test))
])

domain_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=30000,
    sublinear_tf=True
)

X_domain = domain_vec.fit_transform(domain_texts)

domain_clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
domain_clf.fit(X_domain, domain_labels)

# score the held-out validation examples by similarity to test
val_texts = train.set_index("id").loc[
    comparison["id"], "text"
].fillna("")

X_domain_val = domain_vec.transform(val_texts)

analysis = comparison.copy()
analysis["test_likeness"] = domain_clf.predict_proba(
    X_domain_val
)[:, 1]

# compare performance on the most vs. least test-like passages
low = analysis[
    analysis["test_likeness"] <= analysis["test_likeness"].quantile(0.25)
]
high = analysis[
    analysis["test_likeness"] >= analysis["test_likeness"].quantile(0.75)
]

for name, subset in [("Least test-like", low), ("Most test-like", high)]:
    y = subset["label_hybrid"]

    blend = (
        0.25 * subset["hybrid_prob"]
        + 0.75 * subset["selected_probability_ai"]
    )

    print(f"\n{name}: {len(subset)} examples")
    print(f"Hybrid MLP AUROC: {roc_auc_score(y, subset['hybrid_prob']):.6f}")
    print(f"Mia ensemble AUROC: {roc_auc_score(y, subset['selected_probability_ai']):.6f}")
    print(f"Blend AUROC: {roc_auc_score(y, blend):.6f}")



Least test-like: 500 examples
Hybrid MLP AUROC: 0.998194
Mia ensemble AUROC: 0.999303
Blend AUROC: 0.999446

Most test-like: 500 examples
Hybrid MLP AUROC: 0.991746
Mia ensemble AUROC: 0.994391
Blend AUROC: 0.994576


In [47]:
# add Mia's individual model predictions
component_analysis = analysis.merge(
    mia_val[["id", "style", "mlp", "combined"]],
    on="id",
    validate="one_to_one"
)

# identify the most test-like 25% of validation examples
high = component_analysis[
    component_analysis["test_likeness"] >=
    component_analysis["test_likeness"].quantile(0.75)
]

# compare model components on the same examples
models = {
    "Style boosting": "style",
    "Style MLP": "mlp",
    "Lexical LR": "combined",
    "Hybrid MLP": "hybrid_prob",
    "Mia ensemble": "selected_probability_ai"
}

print("Model performance on the 500 most test-like examples:\n")

for name, col in models.items():
    auc = roc_auc_score(high["label_hybrid"], high[col])
    print(f"{name:20s} AUROC: {auc:.6f}")


Model performance on the 500 most test-like examples:

Style boosting       AUROC: 0.992319
Style MLP            AUROC: 0.990853
Lexical LR           AUROC: 0.959319
Hybrid MLP           AUROC: 0.991746
Mia ensemble         AUROC: 0.994391


In [48]:
# Mia's ensemble without the lexical component
style_only_ensemble = (
    0.75 * component_analysis["style"]
    + 0.15 * component_analysis["mlp"]
) / 0.90

# compare blends with and without lexical LR
component_analysis["blend_original"] = (
    0.25 * component_analysis["hybrid_prob"]
    + 0.75 * component_analysis["selected_probability_ai"]
)

component_analysis["blend_no_lexical"] = (
    0.25 * component_analysis["hybrid_prob"]
    + 0.75 * style_only_ensemble
)

# evaluate across all validation examples and test-like examples
subsets = {
    "Full validation": component_analysis,
    "Most test-like 25%": component_analysis[
        component_analysis["test_likeness"] >=
        component_analysis["test_likeness"].quantile(0.75)
    ]
}

for name, subset in subsets.items():
    y = subset["label_hybrid"]

    original_auc = roc_auc_score(y, subset["blend_original"])
    no_lexical_auc = roc_auc_score(y, subset["blend_no_lexical"])

    print(f"\n{name}")
    print(f"Original blend:    {original_auc:.6f}")
    print(f"Without lexical:   {no_lexical_auc:.6f}")
    print(f"Difference:        {no_lexical_auc - original_auc:+.6f}")



Full validation
Original blend:    0.998052
Without lexical:   0.997854
Difference:        -0.000198

Most test-like 25%
Original blend:    0.994576
Without lexical:   0.993919
Difference:        -0.000657


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

# separate the original 8000 training and 2000 validation passages
val_ids = set(comparison["id"])

train_part = train[~train["id"].isin(val_ids)]
val_part = train[train["id"].isin(val_ids)]

assert len(train_part) == 8000
assert len(val_part) == 2000

# character TF-IDF can detect similar wording and templates
similarity_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=30000,
    sublinear_tf=True
)

X_reference = similarity_vec.fit_transform(
    train_part["text"].fillna("")
)

# find each passage's closest match in the training set
def closest_similarity(texts, batch_size=250):
    X = similarity_vec.transform(texts.fillna(""))
    scores = []

    for i in range(0, X.shape[0], batch_size):
        sims = cosine_similarity(
            X[i:i + batch_size], X_reference
        )
        scores.extend(sims.max(axis=1))

    return np.array(scores)

val_similarity = closest_similarity(val_part["text"])
test_similarity = closest_similarity(test["text"])

print("Validation vs. training:")
print(f"Median similarity: {np.median(val_similarity):.4f}")
print(f"Similarity >= 0.90: {(val_similarity >= 0.90).mean():.2%}")

print("\nKaggle test vs. training:")
print(f"Median similarity: {np.median(test_similarity):.4f}")
print(f"Similarity >= 0.90: {(test_similarity >= 0.90).mean():.2%}")


Validation vs. training:
Median similarity: 0.3576
Similarity >= 0.90: 0.20%

Kaggle test vs. training:
Median similarity: 0.2748
Similarity >= 0.90: 0.00%


In [50]:
# attach nearest-training similarity to each validation prediction
similarity_df = pd.DataFrame({
    "id": val_part["id"].to_numpy(),
    "similarity": val_similarity
})

similarity_analysis = comparison.merge(
    similarity_df,
    on="id",
    validate="one_to_one"
)

# separate validation passages into similarity quartiles
similarity_analysis["group"] = pd.qcut(
    similarity_analysis["similarity"],
    q=4,
    labels=["Lowest", "Low", "High", "Highest"]
)

# compare blended AUROC across similarity groups
for group, subset in similarity_analysis.groupby("group", observed=True):
    blend = (
        0.25 * subset["hybrid_prob"]
        + 0.75 * subset["selected_probability_ai"]
    )

    auc = roc_auc_score(subset["label_hybrid"], blend)

    print(
        f"{group:8s} similarity | "
        f"n={len(subset)} | "
        f"median={subset['similarity'].median():.4f} | "
        f"AUROC={auc:.6f}"
    )


Lowest   similarity | n=500 | median=0.2672 | AUROC=0.990686
Low      similarity | n=500 | median=0.3279 | AUROC=0.999775
High     similarity | n=500 | median=0.3949 | AUROC=0.999887
Highest  similarity | n=500 | median=0.4952 | AUROC=0.999434


In [51]:
# focus on validation passages least similar to training
hard_subset = similarity_analysis[
    similarity_analysis["group"] == "Lowest"
].copy()

# calculate 25/75 blended probabilities for these passages
hard_subset["blend_prob"] = (
    0.25 * hard_subset["hybrid_prob"]
    + 0.75 * hard_subset["selected_probability_ai"]
)

# check which class the ensemble struggles with
hard_subset["predicted_label"] = (
    hard_subset["blend_prob"] >= 0.5
).astype(int)

# examine misclassified passages
errors = hard_subset[
    hard_subset["predicted_label"] != hard_subset["label_hybrid"]
]

print("Total examples:", len(hard_subset))
print("Misclassified examples:", len(errors))

print("\nErrors by actual label:")
print(errors["label_hybrid"].value_counts().sort_index())

print("\nConfusion matrix:")
print(pd.crosstab(
    hard_subset["label_hybrid"],
    hard_subset["predicted_label"],
    rownames=["Actual"],
    colnames=["Predicted"]
))


Total examples: 500
Misclassified examples: 18

Errors by actual label:
label_hybrid
0     5
1    13
Name: count, dtype: int64

Confusion matrix:
Predicted    0    1
Actual             
0          266    5
1           13  216


In [52]:
# check whether both models make the same mistakes
errors = hard_subset[
    hard_subset["predicted_label"] != hard_subset["label_hybrid"]
].copy()

errors["hybrid_wrong"] = (
    (errors["hybrid_prob"] >= 0.5).astype(int)
    != errors["label_hybrid"]
)

errors["mia_wrong"] = (
    (errors["selected_probability_ai"] >= 0.5).astype(int)
    != errors["label_hybrid"]
)

print("Blend errors:", len(errors))

print("\nWhich models were wrong on these examples?")
print(pd.crosstab(
    errors["hybrid_wrong"],
    errors["mia_wrong"],
    rownames=["Hybrid wrong"],
    colnames=["Mia wrong"]
))

print("\nBoth models wrong:",
      (errors["hybrid_wrong"] & errors["mia_wrong"]).sum())


Blend errors: 18

Which models were wrong on these examples?
Mia wrong     False  True 
Hybrid wrong              
False             0      6
True              2     10

Both models wrong: 10


## Results and Discussion

The 25% Hybrid MLP + 75% improved ensemble achieved the best validation AUROC among the tested blending weights.

| Model | Validation AUROC |
|---|---:|
| Hybrid MLP | 0.994118 |
| Improved ensemble | 0.997794 |
| 25/75 blend | 0.998052 |

The blended submission achieved a **public Kaggle AUROC of 0.96436**, improving over the Hybrid MLP's 0.96013.

### Generalization Analysis

Adversarial validation achieved 0.7448 AUROC, indicating detectable differences between training and test text distributions.

Kaggle test passages also had lower median character-TF-IDF similarity to training passages (0.2748) than validation passages (0.3576).

The blend performed worse on the least training-like validation examples (0.990686 AUROC), suggesting the random validation split may overestimate performance on unfamiliar writing.

Among 500 low-similarity validation passages, the blend misclassified 18 examples, including 10 where both constituent models were incorrect.

Removing the lexical component reduced validation AUROC, suggesting it contributes complementary information despite weaker standalone performance.

These results are exploratory. The same validation split was used for repeated model-selection experiments, and public Kaggle performance may not represent private leaderboard performance.
